# pii

> whether a document is somebody's business, decided by arithmetic rather than by a model

In [ ]:
#| default_exp pii

The detector is [rahasya](https://vedicreader.github.io/rahasya/): checksummed patterns, 0.996 precision at recall 1.000 (`evals/pii.py`).
What is here is the half that needs a vault: the mark a person sets, and the policy that gates
what retrieval hands back. `ask(pii=...)` defines the modes: see [ask](02_ask.ipynb).


In [ ]:
#| export
from fastcore.all import AttrDict, L, patch
from rahasya import (DENSE, IDENTIFYING, MAX_SCAN, person_spans, pii_report, pii_spans,
                     redact, redact_obj)

# re-exported so a caller holding a Vault has one import, not two
_all_ = ['DENSE', 'IDENTIFYING', 'MAX_SCAN', 'person_spans', 'pii_report', 'pii_spans',
         'redact', 'redact_obj']


## Asking the vault

`Vault.pii` is document-level. `pii_ctx` gates an answer over the sections retrieval chose.

`gated` gates retrieval itself. `ask` used to be the only method taking a policy. Every other one
handed raw section text to whatever read it next. `search`, `context`, `read`, `document`,
`sections`, `toc`, `doc_context` and `federate` all take `pii=` now. The default is `off`.

`redact` masks each identifier. `refuse` replaces the body. Either way the row stays, and `pii`
names the kinds. A caller that asked what matched still learns that something did.

## Learned models are evaluation-only

The ONNX and TFLite detectors do not improve the shipped gate. Their implementations remain in
`evals/backends.py`. `evals/pii_model.py` measures them against the arithmetic detector.
`evals/RESULTS.md` records precision, recall, latency and blind-spot results.


In [ ]:
#| export
from vishalakshi.core import Vault, gate

@patch
def pii(self:Vault,
        ref,                 # a doc_id, source, title or path: whatever `document` takes
        max_chars:int=MAX_SCAN,
        ner:bool=False,      # also look for names, except on code, where identifiers are not names
) -> AttrDict:
    "Whether one whole document is somebody's business, and what in it says so."
    d = self.document(ref, max_chars=max_chars)
    prose = (d.get('kind') or '') != 'code'
    # `scanned_ner` then reports False, which is the honest answer: nothing looked for a name here
    r = pii_report(d.text, ner=ner and prose)
    override = (self.marks(d.get('doc_id')) or {}).get('pii_override') if d.get('doc_id') else None
    r.detected, r.override = r.has_pii, override
    if override == 'clear': r.has_pii = False
    elif override == 'force': r.has_pii = True
    r.doc_id, r.title, r.source = d.get('doc_id'), d.get('title'), d.get('source')
    return r

@patch
def mark_not_pii(self:Vault, ref, clear:bool=True, reason:str='') -> dict:
    "Clear a false-positive PII decision (`pii_override='clear'`), or restore automatic detection."
    return self.mark(ref, pii_override='clear' if clear else None,
                     pii_reason=(reason or None) if clear else None)

@patch
def mark_pii(self:Vault, ref, force:bool=True, reason:str='') -> dict:
    "Force a document private even when arithmetic finds nothing (names, addresses, whole PDFs)."
    return self.mark(ref, pii_override='force' if force else None,
                     pii_reason=(reason or None) if force else None)

def pii_ctx(ctx, ner:bool=False) -> AttrDict:
    "The report for an assembled context, which is what a policy has to gate on."
    parts = [str(getattr(r, 'text', None) or (r.get('text') if isinstance(r, dict) else '') or '')
             for r in (list(ctx.get('results') or []) + list(ctx.get('related') or []))]
    return pii_report('\n\n'.join(parts), ner=ner)

In [ ]:
#| export
GATES = ('off', 'redact', 'refuse')
ROW_TEXT = ('text', 'snippet', 'snippets', 'content', 'summary')
ROW_LABEL = ('title', 'breadcrumb')
ROW_KIDS = ('tree', 'children')
CTX_ROWS = ('results', 'related', 'docs', 'hits')

def _pii_marks(v):
    "Cleared and force-private `(store, doc_id)` pairs, for whichever gate is asking."
    try: rows = list(v._marks()(where="pii_override IN ('clear','force')"))
    except Exception: return set(), set()
    return ({(r['store'], r['doc_id']) for r in rows if r['pii_override'] == 'clear'},
            {(r['store'], r['doc_id']) for r in rows if r['pii_override'] == 'force'})

def _row_get(r, k): return r.get(k) if isinstance(r, dict) else getattr(r, k, None)

def _row_doc(r):
    "The document a row belongs to. A row that names none falls back to its own text."
    # `search` and `context` say `doc_id`, `sections` only `node_id`, `read` says `id`, `federate` `ref`
    for k in ('doc_id', 'node_id', 'id', 'ref'):
        if v := _row_get(r, k): return str(v).split('#', 1)[0]

def _row_kinds(r, cleared, forced, store, ner=False):
    "What one row holds, honouring the mark on its document. `None` when it holds nothing."
    did = _row_doc(r)
    key = (_row_get(r, 'store') or store, did) if did else None
    if key and key in cleared: return None
    if key and key in forced: return {'marked': 1}
    parts = []
    for f in (*ROW_TEXT, *ROW_LABEL):
        v = _row_get(r, f)
        parts += [str(x) for x in v] if isinstance(v, (list, tuple)) else ([str(v)] if v else [])
    rep = pii_report('\n\n'.join(parts), ner=ner)
    return dict(rep.identifying) if rep.has_pii else None

def _section_private(r, cleared, forced, store:str, ner:bool=False) -> bool:
    "Whether one retrieved section is somebody's business. What `ask` filters its context on."
    return bool(_row_kinds(r, cleared, forced, store, ner))

def held(kinds) -> str:
    "What stands in for text a `refuse` policy will not hand back."
    return f"[withheld: personal information ({', '.join(sorted(kinds))})]"

def _mask(r, f, fix):
    "Rewrite field `f` of `r` through `fix`, whether it holds a string or a list of them."
    if not (v := r.get(f)): return
    r[f] = type(v)(fix(str(x)) for x in v) if isinstance(v, (list, tuple)) else fix(str(v))

def _gate_row(r, act, cleared, forced, store, ner):
    "One row masked or withheld, and whatever nests under it. A clean parent can hold a private child."
    if not isinstance(r, dict): return r
    kinds = _row_kinds(r, cleared, forced, store, ner)
    kids = {f: v for f in ROW_KIDS if isinstance(v := r.get(f), (dict, list, tuple, L))}
    if not kinds and not kids: return r
    out = type(r)(r)
    label = lambda s: redact(s, ner=ner)
    if kinds:
        for f in ROW_TEXT: _mask(out, f, label if act == 'redact' else lambda s: held(kinds))
        for f in ROW_LABEL: _mask(out, f, label)
        out['pii'] = kinds
    g = lambda x: _gate_row(x, act, cleared, forced, store, ner)
    for f, v in kids.items(): out[f] = g(v) if isinstance(v, dict) else type(v)(g(k) for k in v)
    return out

def gated(o,                  # whatever a retrieval primitive returned
          pii:str='off',      # off | redact | refuse
          vault=None,         # the shelf whose marks apply; None -> no marks
          ner:bool=False,     # gate on titled names too
          store:str=None,     # the shelf the rows came from; None -> the vault's own
):
    "Apply a retrieval policy to rows, an assembled context, or one section."
    if pii == 'off' or o is None: return o
    if pii not in GATES: raise ValueError(f'unknown pii policy for retrieval: {pii!r}; one of {GATES}')
    cleared, forced = _pii_marks(vault) if vault is not None else (set(), set())
    g = lambda r: _gate_row(r, pii, cleared, forced, store or getattr(vault, 'name', 'store'), ner)
    if isinstance(o, str):
        rep = pii_report(o, ner=ner)
        if not rep.has_pii: return o
        return redact(o, ner=ner) if pii == 'redact' else held(rep.identifying)
    if isinstance(o, (list, tuple, L)): return type(o)(g(r) for r in o)
    if not isinstance(o, dict): return o
    rows = [k for k in CTX_ROWS if isinstance(o.get(k), (list, tuple, L))]
    if not rows and not isinstance(o.get('doc'), dict): return g(o)
    out = type(o)(o)
    for k in rows: out[k] = L(g(r) for r in out[k])
    if isinstance(out.get('doc'), dict): out['doc'] = g(out['doc'])
    return out


In [ ]:
#| export
@patch
@gate
def toc(self:Vault,
        doc=None,            # narrow to one document, as `Index.toc` takes it
        **kw                 # forwarded to `Index.toc`
) -> list:
    "The heading tree. Node titles are the openings of their sections, and a policy covers them too."
    from litesearch import Index
    return Index.toc(self, doc, **kw)

In [ ]:
#| hide
from tempfile import mkdtemp
from pathlib import Path
from fastcore.test import test_eq, test_fail
from vishalakshi import Vault

_g = Vault(str(Path(mkdtemp())/'gate.db'), offline=True)
_g.note('Invoice 4471 for Ada, ada@example.com, phone 020 7946 0958. Card 4111 1111 1111 1111.',
        title='invoice 4471')
_g.note('The deploy pipeline runs on GitHub Actions and takes 20 minutes.', title='pipeline')
_inv, _pipe = _g.doc('invoice 4471')['id'], _g.doc('pipeline')['id']
_LEAK = ('ada@example.com', '4111 1111 1111 1111', '020 7946 0958')

def _leaks(o): return [s for s in _LEAK if s in str(o)]

#: every primitive that hands section text back, and how to call it on this vault
_prims = dict(
    search      = lambda **k: _g.search('invoice 4471', **k),
    sections    = lambda **k: _g.sections('invoice 4471', **k),
    context     = lambda **k: _g.context('invoice 4471', code=0, shelves=0, **k),
    read        = lambda **k: _g.read(f'{_inv}#0', **k),
    document    = lambda **k: _g.document(_inv, **k),
    doc_context = lambda **k: _g.doc_context(_inv, 'invoice', related=0, **k),
)

# `off` is the default: a caller that does not ask gets what it always got
for nm, f in _prims.items(): test_eq((nm, bool(_leaks(f()))), (nm, True))
# ...and neither mode leaves an identifier behind
for act in ('redact', 'refuse'):
    for nm, f in _prims.items(): test_eq((act, nm, _leaks(f(pii=act))), (act, nm, []))

# `local` is `ask`'s: retrieval has no model to send a question to
test_fail(lambda: _g.read(f'{_inv}#0', pii='local'), contains='off')
test_fail(lambda: _g.read(f'{_inv}#0', pii='sortof'), contains='unknown pii policy')

In [ ]:
#| hide
# What the two modes do to one row, and what they leave alone
_row = _g.read(f'{_inv}#0', pii='refuse')
test_eq(_row['text'], '[withheld: personal information (card, email, phone)]')
# the kinds, so a caller can say why. Counted over body and label together, so a note whose title
# repeats its first line reports each kind twice; the kinds are the answer, the tallies are not
test_eq(sorted(_row['pii']), ['card', 'email', 'phone'])
test_eq(_row['title'], 'invoice 4471')                         # a clean label is left as it is
test_eq('[EMAIL]' in _g.read(f'{_inv}#0', pii='redact')['text'], True)

# a title that carries an identifier is masked rather than withheld, under either mode: a note is
# titled by its own first line, and a row with no label left cannot be opened or asked about
_titled = dict(doc_id='d1', title='Invoice for ada@example.com', text='Invoice for ada@example.com')
for _act in ('redact', 'refuse'):
    _t = gated([_titled], _act)[0]
    test_eq(_t['title'], 'Invoice for [EMAIL]')
    test_eq(_leaks(_t), [])
test_eq(gated([_titled], 'refuse')[0]['text'], '[withheld: personal information (email)]')

# `summary` is a sentence of the section, so it holds whatever the section holds
for _f in ('text', 'summary'): assert not _leaks(_row.get(_f)), _f

# a clean row is returned untouched, and carries no verdict
_clean = [r for r in _g.search('deploy pipeline', pii='refuse') if 'GitHub' in str(r['snippet'])]
test_eq(len(_clean), 1)
test_eq('pii' in _clean[0], False)

# the marks reach every primitive, not just `ask`
_g.mark_not_pii(_inv, reason='my own invoice')
for nm, f in _prims.items(): test_eq((nm, bool(_leaks(f(pii='refuse')))), (nm, True))
_g.mark_not_pii(_inv, clear=False)
test_eq(_leaks(_g.read(f'{_inv}#0', pii='refuse')), [])

# ...including on a document arithmetic finds nothing in
test_eq('GitHub' in str(_g.read(f'{_pipe}#0', pii='refuse')), True)
_g.mark_pii(_pipe, reason='confidential')
_forced = _g.read(f'{_pipe}#0', pii='refuse')
test_eq(('GitHub' in str(_forced), _forced['pii']), (False, {'marked': 1}))
_g.mark_pii(_pipe, force=False)

# a row whose document cannot be named is still gated, on its text alone
test_eq(_leaks(gated([dict(text=_LEAK[0])], 'refuse')), [])
test_eq(gated('nothing identifying', 'refuse'), 'nothing identifying')
test_eq(gated(None, 'refuse'), None)

In [ ]:
#| hide
from tempfile import mkdtemp
from pathlib import Path
from vishalakshi import Vault

v = Vault(Path(mkdtemp())/'p.db', offline=True)
v.add('A letter about Jane, and what she said on Tuesday.', title='letter', source='/inbox/letter.md')
r = v.pii('/inbox/letter.md')
test_eq(r.has_pii, False)
v.mark_pii('/inbox/letter.md', reason='address book')
test_eq(v.pii('/inbox/letter.md').has_pii, True)
test_eq(v.pii('/inbox/letter.md').override, 'force')
v.mark_not_pii('/inbox/letter.md', reason='my own draft')
test_eq(v.pii('/inbox/letter.md').has_pii, False)
test_eq(v.pii('/inbox/letter.md').override, 'clear')

In [ ]:
#| hide
_sig = 'Dr Charles Babbage signed it.'
# 3. no NER on code: an identifier is not a name, and a report must not claim it looked
v.add('# Dr Charles Babbage wrote this\ndef f(): pass', title='mod', source='/m.py', kind='code')
_c = v.pii('/m.py', ner=True)
test_eq((_c.scanned_ner, _c.has_pii), (False, False))

v.add(_sig, title='signed', source='/inbox/signed.md')
_p = v.pii('/inbox/signed.md', ner=True)
test_eq((_p.scanned_ner, _p.has_pii, _p.identifying), (True, True, {'person': 1}))
test_eq(v.pii('/inbox/signed.md').scanned_ner, False)   # the default is still arithmetic only


In [ ]:
#| hide
import inspect

for _fn in (pii_spans, pii_report, redact, redact_obj, Vault.pii, pii_ctx):
    test_eq('model' in inspect.signature(_fn).parameters, False)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()